# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs. We will enumerate all record sets, their fields, and the corresponding `@id`s as defined in the Croissant schema.

In [ ]:
# List all record sets with their @id
record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}")
record_set_ids = []
for idx, recset in enumerate(record_sets):
    print(f"[{idx}] Record set name: {getattr(recset, 'name', None)}, @id: {getattr(recset, '@id', None)}")
    record_set_ids.append(getattr(recset, '@id', None))
    if hasattr(recset, 'fields'):
        for f in recset.fields:
            print(f"      Field: {getattr(f, 'name', None)}, @id: {getattr(f, '@id', None)}, type: {getattr(f, 'data_type', None)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}

# We use the first record set for demonstration, but you can update this to any available @id
record_sets_to_extract = record_set_ids
for rec_id in record_sets_to_extract:
    records = list(dataset.records(record_set=rec_id))
    if records:
        dataframes[rec_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set '{rec_id}' (columns: {list(dataframes[rec_id].columns)})\n")
    else:
        print(f"No records found for record set '{rec_id}'")

# If data was loaded, display the columns and head of the first one
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]  # Pick the first one
    print(f"Columns in {main_record_set_id}: {dataframes[main_record_set_id].columns.tolist()}")
    dataframes[main_record_set_id].head()
else:
    print("No record sets loaded as DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, let's pick a numeric field for analysis.

numeric_field = None
if dataframes:
    df = dataframes[main_record_set_id]
    # Guess numeric columns - pick the first float or int type column
    for col in df.columns:
        # infer the real numeric columns
        try:
            col_data = pd.to_numeric(df[col], errors='coerce')
            if col_data.notna().sum() > 0 and np.issubdtype(col_data.dtype, np.number):
                numeric_field = col
                break
        except Exception:
            continue
    if numeric_field:
        print(f"Selected numeric field for analysis: {numeric_field}")
        # Clean numeric field
        df[numeric_field + "_num"] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field + "_num"].mean()
        filtered_df = df[df[numeric_field + "_num"] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field + "_num"] - filtered_df[numeric_field + "_num"].mean()) / filtered_df[numeric_field + "_num"].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try group by first string/categorical column (different from numeric column)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field + "_num"].mean().reset_index()
            print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("No DataFrame available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below, we plot the distribution of the selected numeric field and, if available, its grouping by a categorical field.

In [ ]:
if dataframes and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field + "_num"].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If we have a group_field, show bar plot
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field + '_num')
        plt.title(f"Mean '{numeric_field}' by '{group_field}'")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable data for plotting.")

## 6. Conclusion
This notebook demonstrated how to load the FAIR² 'Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells' dataset using the `mlcroissant` library, review its structure, extract data, perform common exploratory transformations, and plot summary statistics.

**Next steps:**
- Inspect and explore more `@id`s, fields, and relationships as needed.
- Adapt filtering, grouping, and EDA logic depending on dataset specifics and research questions.

**References:**
- [FAIR² Dataset Metadata](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- [`mlcroissant` documentation](https://mlcommons.github.io/croissant/python/)
